# Sprint 5

## Install PySpark

In [1]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## Spark session

In [2]:
import os
import platform

if platform.system() == "Windows":
    # Point to Java 17 explicitly — required for PySpark on Windows
    os.environ["JAVA_HOME"] = r"C:\Users\bhoom\AppData\Local\Programs\Microsoft\jdk-17.0.18.8-hotspot"
    os.environ["PATH"] = os.environ["JAVA_HOME"] + r"\bin;" + os.environ["PATH"]
    print("Windows: Java 17 path set to", os.environ["JAVA_HOME"])
else:
    print("Non-Windows: no fix needed")

Windows: Java 17 path set to C:\Users\bhoom\AppData\Local\Programs\Microsoft\jdk-17.0.18.8-hotspot


In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131_bladdards")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 3.5.1
Shuffle partitions: 8


## Import the funtions and create data path

In [4]:
from pathlib import Path
from urllib.request import urlretrieve # to download data if not already present

from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    broadcast
)

# Path for MIMIC-IV data
DATA_DIR = Path("data/MIMIC-IV/hosp")


## Dataframes

In [5]:
# -------------------------------------------------------
# Visits dataframe
# -------------------------------------------------------


In [6]:
# -------------------------------------------------------
# BHOOMIKA'S SECTION — diagnoses DataFrame
# Collects all diagnoses for visits of interest
# (pre-BC symptom visits + first BC diagnosis visits)
# -------------------------------------------------------

EVIDENCE_DIR = Path("out/evidence")
# DATA_DIR is already defined above as Path("data/MIMIC-IV/hosp")

# STEP 1: Load visits of interest from pre_bc_symptom_timeline
# row_type (SYMPTOM or BC_FIRST_DX) becomes visit_type
timeline_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(EVIDENCE_DIR / "pre_bc_symptom_timeline.csv"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("row_type").alias("visit_type")
    )
    .dropDuplicates(["hadm_id"])  # one visit_type label per admission
)

print("Timeline visits of interest:", timeline_df.count())
timeline_df.show(5)

# STEP 2: Load all diagnoses from MIMIC
# seq_num = order diagnoses were recorded per visit → becomes ranking
dx_icd_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "diagnoses_icd.csv.gz"))
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("seq_num").cast("int").alias("ranking"),
        col("icd_code"),
        col("icd_version").cast("int")
    )
)

print("diagnoses_icd rows:", dx_icd_df.count())
dx_icd_df.show(5)

# STEP 3: Load ICD code dictionary
# Maps icd_code + icd_version → human readable description
icd_dict_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(str(DATA_DIR / "d_icd_diagnoses.csv.gz"))
    .select(
        col("icd_code"),
        col("icd_version").cast("int"),
        col("long_title").alias("icd_desc")
    )
)

print("ICD dictionary rows:", icd_dict_df.count())
icd_dict_df.show(5)

# STEP 4: Filter diagnoses to visits of interest only
# Inner join on hadm_id — keeps only admissions in our timeline
# broadcast(timeline_df) since it is small (1568 rows vs 6M+)
filtered_dx_df = dx_icd_df.join(
    broadcast(timeline_df),
    on="hadm_id",
    how="inner"
)

print("Diagnoses for visits of interest:", filtered_dx_df.count())

# Drop duplicate subject_id introduced by the join
# (both dx_icd_df and timeline_df have subject_id)
filtered_dx_df2 = filtered_dx_df.drop(timeline_df["subject_id"])

# STEP 5: Enrich with ICD descriptions
# Left join on icd_code + icd_version — must match both since
# same code can mean different things in ICD-9 vs ICD-10
diagnoses = (
    filtered_dx_df2.join(
        broadcast(icd_dict_df),  # dictionary is small, broadcast it
        on=["icd_code", "icd_version"],
        how="left"  # keep all rows even if no dictionary entry found
    )
    .select(
        col("subject_id").cast("int"),
        col("hadm_id").cast("int"),
        col("visit_type").cast("string"),
        col("ranking").cast("int"),
        col("icd_code").cast("string"),
        col("icd_version").cast("int"),
        col("icd_desc").cast("string")
    )
    .orderBy("subject_id", "hadm_id", "ranking")
)

print("=== diagnoses DataFrame ===")
print("Row count:", diagnoses.count())
diagnoses.printSchema()
diagnoses.show(10, truncate=False)


Timeline visits of interest: 1568
+----------+--------+-----------+
|subject_id| hadm_id| visit_type|
+----------+--------+-----------+
|  10383113|20007405|BC_FIRST_DX|
|  13470381|20010741|BC_FIRST_DX|
|  15129856|20010894|BC_FIRST_DX|
|  13349232|20015647|BC_FIRST_DX|
|  15764116|20020797|    SYMPTOM|
+----------+--------+-----------+
only showing top 5 rows

diagnoses_icd rows: 6364488
+----------+--------+-------+--------+-----------+
|subject_id| hadm_id|ranking|icd_code|icd_version|
+----------+--------+-------+--------+-----------+
|  10000032|22595853|      1|    5723|          9|
|  10000032|22595853|      2|   78959|          9|
|  10000032|22595853|      3|    5715|          9|
|  10000032|22595853|      4|   07070|          9|
|  10000032|22595853|      5|     496|          9|
+----------+--------+-------+--------+-----------+
only showing top 5 rows

ICD dictionary rows: 112107
+--------+-----------+--------------------+
|icd_code|icd_version|            icd_desc|
+------

In [7]:
# -------------------------------------------------------
# Aggregation
# -------------------------------------------------------


## Clean up
Stop spark session when done

In [8]:
# Uncomment when you are completely done:

# spark.stop()